## Documentation

To read more about the create index API, visit the [docs](https://www.elastic.co/guide/en/elasticsearch/reference/current/indices-create-index.html).

No elastic cada documento é convertido em  json 
E desse modo criamos um product index 


## Connect to ElasticSearch

In [1]:
from pprint import pprint
from elasticsearch import Elasticsearch

es = Elasticsearch('http://localhost:9200')
client_info = es.info()
print('Connected to Elasticsearch!')
pprint(client_info.body)

Connected to Elasticsearch!
{'cluster_name': 'meu-cluster-local',
 'cluster_uuid': 'BNaV0P0MSwm0L15QURbxRg',
 'name': 'es01',
 'tagline': 'You Know, for Search',
 'version': {'build_date': '2024-12-11T12:08:05.663969764Z',
             'build_flavor': 'default',
             'build_hash': '2b6a7fed44faa321997703718f07ee0420804b41',
             'build_snapshot': False,
             'build_type': 'docker',
             'lucene_version': '9.12.0',
             'minimum_index_compatibility_version': '7.0.0',
             'minimum_wire_compatibility_version': '7.17.0',
             'number': '8.17.0'}}


## Create index

### 1. Simplest way

In this method, the `mappings` which define the structure of documents within an index are infered automatically

In [3]:
es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(index='my_index')

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_index'})

### 2. Specify the number of replicas and shards

`Shards`: O Elasticsearch divide os dados de um índice em vários shards. Cada shard é um índice independente que o Elasticsearch pode distribuir entre vários nós em um cluster. Os shards são gerenciados automaticamente, mas são configurados no momento da criação do índice.

`Replicas`: Para tolerância a falhas e alta disponibilidade, um índice pode ter shards de réplica, que são cópias dos shards primários.

Supondo que tenha 3 arquivos e Shards é igual a 2. Cada documento é dividido em 2 partes. Assim o product index conterá 6 partes em vez de 3 documentos inteios 


Numero de replicas significa que duplicar ou triplicar os documentos para aumentar a reseliência 
Além disso réplicas permitem buscas paralelas em operações de recuperação 


In [4]:
es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(
    index="my_index",
    settings={
        "index": {
            "number_of_shards": 3,  # how many pieces the data is split into
            "number_of_replicas": 2  # how many copies of the data
        }
    },
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_index'})


## Insert one document

Create a dummy index just to test inserting one document


In [5]:
es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(index='my_index')

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_index'})

In [6]:
document = {
    'title': 'title',
    'text': 'text',
    'created_on': '2024-09-22',
}
response = es.index(index='my_index', body=document)
response

ObjectApiResponse({'_index': 'my_index', '_id': '_T92BJ4Bi2gNu3tEx3LT', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1})

O objeto de resposta contém o resultado da operação. Se inserirmos o documento com sucesso, então result = created. Cada documento possui um id e é fragmentado em shards.

In [7]:
print(response["result"])

created


In [8]:
print(response["_shards"])

{'total': 2, 'successful': 1, 'failed': 0}


### Insert multiple documents
Just do the same step but in a for loop


In [9]:
import json
dummy_data = json.load(open("data/dummy_data.json"))
dummy_data

[{'title': 'Sample Title 1',
  'text': 'This is the first sample document text.',
  'created_on': '2024-09-22'},
 {'title': 'Sample Title 2',
  'text': 'Here is another example of a document.',
  'created_on': '2024-09-24'},
 {'title': 'Sample Title 3',
  'text': 'The content of the third document goes here.',
  'created_on': '2024-09-24'}]

In [10]:
def insert_document(document):
    response = es.index(index='my_index', body=document)
    return response


def print_info(response):
    print(f"""Document ID: {response['_id']} is '{
          response["result"]}' and is split into {response['_shards']['total']} shards.""")


for document in dummy_data:
    response = insert_document(document)
    print_info(response)

Document ID: _j95BJ4Bi2gNu3tE1HJ- is 'created' and is split into 2 shards.
Document ID: _z95BJ4Bi2gNu3tE1HKV is 'created' and is split into 2 shards.
Document ID: AD95BJ4Bi2gNu3tE1HOo is 'created' and is split into 2 shards.


## Print mapping

In [11]:
from pprint import pprint

index_mapping = es.indices.get_mapping(index='my_index')
pprint(index_mapping["my_index"]["mappings"]["properties"])

{'created_on': {'type': 'date'},
 'text': {'fields': {'keyword': {'ignore_above': 256, 'type': 'keyword'}},
          'type': 'text'},
 'title': {'fields': {'keyword': {'ignore_above': 256, 'type': 'keyword'}},
           'type': 'text'}}


## Manual MApping

In [12]:
es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(index='my_index')

mapping = {
    'properties': {
        'created_on': {'type': 'date'},
        'text': {
            'type': 'text',
            'fields': {
                'keyword': {
                    'type': 'keyword',
                    'ignore_above': 256
                }
            }
        },
        'title': {
            'type': 'text',
            'fields': {
                'keyword': {
                    'type': 'keyword',
                    'ignore_above': 256
                }
            }
        }
    }
}

es.indices.put_mapping(index='my_index', body=mapping)

index_mapping = es.indices.get_mapping(index='my_index')
pprint(index_mapping["my_index"]["mappings"]["properties"])


{'created_on': {'type': 'date'},
 'text': {'fields': {'keyword': {'ignore_above': 256, 'type': 'keyword'}},
          'type': 'text'},
 'title': {'fields': {'keyword': {'ignore_above': 256, 'type': 'keyword'}},
           'type': 'text'}}


In [13]:
mapping = {
    'properties': {
        'created_on': {'type': 'date'},
        'text': {
            'type': 'text',
            'fields': {
                'keyword': {
                    'type': 'keyword',
                    'ignore_above': 256
                }
            }
        },
        'title': {
            'type': 'text',
            'fields': {
                'keyword': {
                    'type': 'keyword',
                    'ignore_above': 256
                }
            }
        }
    }
}

es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(index='my_index', mappings=mapping)

index_mapping = es.indices.get_mapping(index='my_index')
pprint(index_mapping["my_index"]["mappings"]["properties"])

{'created_on': {'type': 'date'},
 'text': {'fields': {'keyword': {'ignore_above': 256, 'type': 'keyword'}},
          'type': 'text'},
 'title': {'fields': {'keyword': {'ignore_above': 256, 'type': 'keyword'}},
           'type': 'text'}}
